# Week 04 Walkthrough — Mushroom RF (4.1) + Bank Logistic (4.2)

**Vault note:** `obsidian/02 Assignments/Week04 Logistic Regression and Random Forest.md`

สองงานใน `attachment_week4/`:
- **4.1** `student.ipynb` — Mushroom + Random Forest + GridSearchCV
- **4.2** `student.py` — Bank marketing + Logistic Regression

---
## Part A — 4.1 Mushroom Random Forest

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

### A1. โหลด + Q1 — missing ใน `gill-size`

ก่อน prep นับ NaN (ไม่ใช่ string `'na'`)

In [ ]:
DATA = '4.1/mushroom2020_dataset.csv'
df = pd.read_csv(DATA)
print('rows:', len(df))
print('gill-size missing (Q1):', df['gill-size'].isna().sum())

### A2. Drop label หาย + ลบคอลัมน์ (Q2)

In [ ]:
DROP_COLS = [
    'id', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color-rate',
    'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring',
    'stalk-color-above-ring-rate', 'stalk-color-below-ring-rate',
    'veil-color-rate', 'veil-type',
]

df = df.dropna(subset=['label'])
df = df.drop(columns=DROP_COLS)
print('Q2 shape:', df.shape)
print('still missing:', df.isna().sum().sum())

### A3. Impute + label (Q3)

โจทย์ให้ impute ก่อน split — ใช้ mean/mode จากทั้งชุด

In [ ]:
num_cols = df.select_dtypes(include='number').columns
cat_cols = [c for c in df.select_dtypes(exclude='number').columns if c != 'label']

for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

y = df['label'].map({'p': 0, 'e': 1})
X = df.drop(columns=['label'])
print('Q3 class0:class1 =', (y == 0).sum(), ':', (y == 1).sum())

### A4. One-hot + split (Q4)

In [ ]:
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2020, stratify=y
)
print('Q4 train:test =', X_train.shape[0], X_test.shape[0])

### A5. GridSearch RF (Q5–Q6)

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3],
    'min_samples_leaf': [2, 5],
    'n_estimators': [100],
    'random_state': [2020],
}
gs = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='f1_weighted')
gs.fit(X_train, y_train)
print('Q5 best:', gs.best_params_)

pred = gs.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))
print('Q6 macro F1:', round(f1_score(y_test, pred, average='macro'), 2))

---
## Part B — 4.2 Bank Logistic Regression

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

bank = pd.read_csv('4.2/bank-st.csv')
print('Q1 rows:', len(bank))
print('Q3 no:yes =', round((bank['y']=='no').mean(), 3), round((bank['y']=='yes').mean(), 3))

### B1. Dedupe + unknown + split (Q4–Q5)

In [ ]:
df = bank.drop_duplicates().replace('unknown', np.nan)
print('Q4 shape:', df.shape)

flat_cols = [
    c for c in df.columns
    if c != 'y' and df[c].value_counts(normalize=True, dropna=False).max() > 0.99
]
print('flat cols dropped:', flat_cols)

y = (df['y'] == 'yes').astype(int)
X = df.drop(columns=['y'])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)
print('Q5 shapes:', X_train.shape, X_test.shape)

### B2. Impute จาก train + encode (Q6)

ต่างจาก Mushroom — สถิติ impute ต้องมาจาก train เท่านั้น

In [ ]:
EDUCATION_ORDER = {
    'illiterate': 1, 'basic.4y': 2, 'basic.6y': 3, 'basic.9y': 4,
    'high.school': 5, 'professional.course': 6, 'university.degree': 7,
}

X_train = X_train.copy()
X_test = X_test.copy()
for col in X_train.select_dtypes(include='number').columns:
    fill = X_train[col].mean()
    X_train[col] = X_train[col].fillna(fill)
    X_test[col] = X_test[col].fillna(fill)
for col in X_train.select_dtypes(exclude='number').columns:
    fill = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(fill)
    X_test[col] = X_test[col].fillna(fill)

X_train['education'] = X_train['education'].map(EDUCATION_ORDER)
X_test['education'] = X_test['education'].map(EDUCATION_ORDER)
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
print('Q6 X_train shape:', X_train.shape)

### B3. Logistic Regression (Q7)

In [ ]:
model = LogisticRegression(
    random_state=2025, class_weight='balanced', max_iter=500
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Q7 macro F1:', round(f1_score(y_test, pred, average='macro'), 3))